# 📊 Analiza metryk DQN — Porównanie runów treningowych

Notebook do analizy logów treningowych i ewaluacyjnych zapisanych w CSV.

**Zawiera:**
1. Import bibliotek (Pandas, CSV, Matplotlib)
2. Wczytywanie danych — moduł `csv` vs `pandas`
3. Statystyki i diagnostyka runów
4. Rolling average reward
5. Porównanie wielu runów na wykresach
6. Eksport podsumowania

**Wymagania:** `pandas`, `numpy`, `matplotlib`

## 1. Import bibliotek

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Moduł analizy z projektu DQN Framework
from utils.analyze import (
    list_runs,
    load_run,
    load_latest,
    compare_runs,
    run_summary,
    diagnose,
    build_summary_report,
    export_summary_report,
)

METRICS_DIR = Path("metrics")
COLORS = plt.cm.tab10.colors

print("Dostępne pliki CSV:")
for f in sorted(METRICS_DIR.glob("*.csv")):
    print(f"  {f.name}")

## 2. Wczytywanie danych CSV — moduł `csv` vs `pandas`

Porównanie dwóch podejść do wczytywania danych z plików CSV.

In [ ]:
# --- Podejście 1: moduł csv (standardowa biblioteka) ---
# Przydatne gdy nie chcesz zależności od pandas

runs = list_runs()
if not runs.empty:
    sample_file = runs.iloc[-1]["path"]
    print(f"Wczytywanie: {runs.iloc[-1]['file']}\n")

    rows = []
    with open(sample_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)

    print(f"Wczytano {len(rows)} wierszy (moduł csv)")
    print("Pierwsze 3 wiersze:")
    for r in rows[:3]:
        print(f"  {dict(r)}")
else:
    print("Brak plików CSV w metrics/")

In [ ]:
# --- Podejście 2: pandas (rekomendowane) ---

if not runs.empty:
    df = load_run(runs.iloc[-1]["path"])
    print(f"Wczytano: {runs.iloc[-1]['file']}")
    print(f"Shape: {df.shape}\n")
    display(df.head())
    print()
    df.describe()

## 3. Przegląd dostępnych runów

Lista wszystkich runów treningowych i ewaluacyjnych z metadanymi.

In [ ]:
# Wszystkie dostępne runy
all_runs = list_runs()
if all_runs.empty:
    print("Brak plików CSV w metrics/.")
else:
    display(all_runs[["file", "env", "model", "timestamp", "type"]])

print(f"\nŁącznie: {len(all_runs)} plików CSV")
print(f"  Training: {len(all_runs[all_runs['type'] == 'train'])}")
print(f"  Eval:     {len(all_runs[all_runs['type'] == 'eval'])}")

## 4. Porównanie runów treningowych

Tabela porównawcza dla wybranego środowiska — zmień `ENV` poniżej.

In [ ]:
# Zmień ENV na interesujące środowisko
ENV = "CartPole-v1"  # "MountainCar-v0" | "Acrobot-v1"

print(f"=== {ENV} — Training Runs ===\n")
cmp_train = compare_runs(ENV, "train")
if not cmp_train.empty:
    display(cmp_train)
else:
    print("Brak runów treningowych.")

print(f"\n=== {ENV} — Eval Runs ===\n")
cmp_eval = compare_runs(ENV, "eval")
if not cmp_eval.empty:
    display(cmp_eval)
else:
    print("Brak runów ewaluacyjnych.")

## 5. Statystyki i diagnostyka — ostatni run

Oblicza podstawowe statystyki i automatycznie diagnozuje problemy treningowe.

In [ ]:
df, meta = load_latest(ENV, "train")

if df is not None:
    print(f"Run: {meta['file']}")
    print(f"Epizody: {len(df)}\n")

    print("=== Statystyki total_reward ===")
    stats = df["reward"].describe()
    print(f"  Średnia:   {stats['mean']:.2f}")
    print(f"  Std:       {stats['std']:.2f}")
    print(f"  Min:       {stats['min']:.2f}")
    print(f"  Max:       {stats['max']:.2f}")

    best_ep = df.loc[df["reward"].idxmax(), "episode"]
    print(f"  Najlepszy epizod: #{int(best_ep)}")

    print(f"\n=== Epsilon ===")
    print(f"  Start:   {df['epsilon'].iloc[0]:.4f}")
    print(f"  Połowa:  {df['epsilon'].iloc[len(df)//2]:.4f}")
    print(f"  Koniec:  {df['epsilon'].iloc[-1]:.4f}")

    print(f"\n=== Diagnostyka automatyczna ===")
    for obs in diagnose(ENV):
        print(f"  • {obs}")
else:
    print("Brak danych.")

## 6. Rolling average reward

`df['reward'].rolling(window=100).mean()` — porównanie z avg100 zapisanym przez train.py.

In [ ]:
if df is not None:
    WINDOW = 100

    # Rolling average obliczona tutaj (identyczna z avg100 z train.py gdy min_periods=1)
    df["rolling_avg"] = df["reward"].rolling(window=WINDOW, min_periods=1).mean()

    print("Porównanie rolling_avg (obliczone teraz) vs avg100 (zapisane podczas treningu):")
    print(df[["episode", "reward", "rolling_avg", "avg100"]].tail(10).to_string(index=False))

    diff = (df["rolling_avg"] - df["avg100"]).abs().max()
    print(f"\nMaks. różnica: {diff:.6f}  {'✓ identyczne' if diff < 1e-3 else '⚠ różnice'}")

## 7. Porównanie wielu runów — wykresy

Nakłada przebiegi avg100, rolling reward i epsilon wszystkich runów danego środowiska.

In [ ]:
train_runs = list_runs(env_name=ENV, train_only=True).sort_values("timestamp")

if train_runs.empty:
    print(f"Brak runów treningowych dla {ENV}.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=False)
    fig.suptitle(f"{ENV} — Porównanie runów treningowych", fontsize=14, fontweight="bold")

    for i, (_, run) in enumerate(train_runs.iterrows()):
        df_run = load_run(run["path"])
        label = f"{run['model']}  {run['timestamp'].strftime('%m-%d %H:%M')}"
        color = COLORS[i % len(COLORS)]
        alpha_raw = 0.25

        # Wykres 1: reward surowy + rolling avg
        axes[0].plot(df_run["episode"], df_run["reward"], alpha=alpha_raw, color=color, linewidth=0.7)
        axes[0].plot(
            df_run["episode"],
            df_run["reward"].rolling(100, min_periods=1).mean(),
            label=label, color=color, linewidth=1.8,
        )

        # Wykres 2: avg100 (zapisane przez train.py)
        axes[1].plot(df_run["episode"], df_run["avg100"], label=label, color=color, linewidth=1.8)

        # Wykres 3: epsilon
        axes[2].plot(df_run["episode"], df_run["epsilon"], label=label, color=color, linewidth=1.4)

    axes[0].set_title("Reward (surowy + rolling avg 100)")
    axes[0].set_ylabel("Reward")
    axes[0].legend(fontsize=7, loc="upper left")
    axes[0].grid(alpha=0.3)

    axes[1].set_title("Avg100 (zapisane przez train.py)")
    axes[1].set_ylabel("Avg100")
    axes[1].legend(fontsize=7, loc="upper left")
    axes[1].grid(alpha=0.3)

    axes[2].set_title("Epsilon — tempo eksploracji")
    axes[2].set_xlabel("Epizod")
    axes[2].set_ylabel("Epsilon")
    axes[2].legend(fontsize=7, loc="upper right")
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

## 8. Wykres eval/mean_reward

Greedy evaluation (epsilon=0) — postęp metryki sukcesu przez kolejne ewaluacje w trakcie treningu.

In [ ]:
eval_runs = list_runs(env_name=ENV, eval_only=True).sort_values("timestamp")

if eval_runs.empty:
    print(f"Brak eval CSV dla {ENV}. Uruchom trening — eval generuje się automatycznie co 100 epizodów.")
else:
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.set_title(f"{ENV} — Eval mean_reward (greedy policy, ε=0)", fontsize=13, fontweight="bold")

    for i, (_, run) in enumerate(eval_runs.iterrows()):
        df_eval = load_run(run["path"])
        label = f"{run['model']}  {run['timestamp'].strftime('%m-%d %H:%M')}"
        color = COLORS[i % len(COLORS)]

        ax.plot(df_eval["episode"], df_eval["mean_reward"], label=label, color=color, linewidth=2)
        if "std_reward" in df_eval.columns:
            ax.fill_between(
                df_eval["episode"],
                df_eval["mean_reward"] - df_eval["std_reward"],
                df_eval["mean_reward"] + df_eval["std_reward"],
                alpha=0.1, color=color,
            )

    ax.set_xlabel("Epizod treningowy")
    ax.set_ylabel("Mean reward (greedy)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 9. Eksport podsumowania do CSV

Tworzy tabelę porównawczą runów i zapisuje ją do `metrics/runs_summary_<env>.csv`.

In [ ]:
summary, output_path = export_summary_report(ENV)

if summary.empty:
    print("Brak danych do eksportu.")
else:
    print(f"Zapisano: {output_path}")
    display(summary)

    best_col = "best_mean_reward" if "best_mean_reward" in summary.columns else "best_avg100"
    if best_col in summary.columns:
        best_row = summary.loc[summary[best_col].idxmax()]
        print(f"\nNajlepszy run wg {best_col}: {best_row['model']} | {best_row['timestamp']}")